# Circuit creation function

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, open_docs

from qiskit_metal.qlibrary.qubits.transmon_pocket_6 import TransmonPocket6
from qiskit_metal.qlibrary.qubits.transmon_cross import TransmonCross
from qiskit_metal.qlibrary.qubits.transmon_cross_fl import TransmonCrossFL

from qiskit_metal.qlibrary.couplers.tunable_coupler_01 import TunableCoupler01

from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight

from qiskit_metal.qlibrary.lumped.cap_n_interdigital import CapNInterdigital
from qiskit_metal.qlibrary.couplers.cap_n_interdigital_tee import CapNInterdigitalTee
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee

from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.terminations.launchpad_wb_coupled import LaunchpadWirebondCoupled
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround

from qiskit_metal.qlibrary.qubits.JJ_Manhattan import jj_manhattan

design = metal.designs.DesignPlanar()

gui = metal.MetalGUI(design)

design.overwrite_enabled = True

# board size and cpw configuration
design._chips['main']['size']['size_x'] = '5mm'
design._chips['main']['size']['size_y'] = '5mm'
design.variables['cpw_width'] = '10 um'
design.variables['cpw_gap'] = '6 um'

xmon_options = dict(
    pos_x = '1.0mm',
    pos_y = '4.25mm',
    orientation = '270',
    cross_width = '100um',
    connection_pads=dict(
        connector_1 = dict(connector_location = '90', connector_type = '0')
    ),
)

Qubit_1 = TransmonCross(design, 'Qubit_1', options=xmon_options)

gui.rebuild()

output = LaunchpadWirebondCoupled(design, 'output', options = dict(pos_x='2500um', pos_y='260um', orientation='90', lead_length='30um'))
input = LaunchpadWirebondCoupled(design, 'input', options = dict(pos_x='2500um', pos_y='4740um', orientation='270', lead_length='30um'))
gui.rebuild()

IObus = RouteStraight(design,'IObus',options=Dict(pin_inputs=Dict(
start_pin=Dict(
        component = 'input',
        pin = 'tie'),
    end_pin=Dict(
        component = 'output',
        pin = 'tie')
)))
gui.rebuild()

pos_ro_x = 2500
cpw_width = 10
epsilon = 6.3
fillet='74.99um'
cpw_options = Dict(
    lead=Dict(
        start_straight='200um',
        end_straight='200um'
    ),
    fillet=fillet,
    meander=Dict(spacing="150um")
)


def pos_from_offset(offset):
    return pos_ro_x + offset

def quarter_wave_length(frequency_ghz, epsilon_eff):
    """input in GHz, output in um"""
    c = 3e8
    frequency_hz = frequency_ghz * 1e9
    wavelength = c / (frequency_hz * (epsilon_eff ** 0.5))
    quarter_wavelength = wavelength / 4
    return quarter_wavelength * 1e6

def generate_readout_frequencies(center_freq_ghz=7.4, spacing_mhz=30, num_qubits=6):
    start_freq = center_freq_ghz - (spacing_mhz * (num_qubits - 1) / 2) / 1000
    return [round(start_freq + i * spacing_mhz / 1000, 6) for i in range(num_qubits)]

def connect(cpw_name: str, pin1_comp_name: str, pin1_comp_pin: str, pin2_comp_name: str, pin2_comp_pin: str,
            length: str, asymmetry='0 um'):
    """Connect two pins with a CPW."""
    myoptions = Dict(
        pin_inputs=Dict(
            start_pin=Dict(
                component=pin1_comp_name,
                pin=pin1_comp_pin),
            end_pin=Dict(
                component=pin2_comp_name,
                pin=pin2_comp_pin)),
        total_length=length)
    myoptions.update(cpw_options)
    myoptions.meander.asymmetry = asymmetry
    return RouteMeander(design, cpw_name, myoptions)

frequencies = generate_readout_frequencies()

length_um_list = [quarter_wave_length(freq, epsilon) for freq in frequencies]

offset_ro_x_1 = -40
pos_x_cp1 = pos_from_offset(offset_ro_x_1)
pos_y_cp1 = 4250

coupling_pin_1 = OpenToGround(design, 'coupling_pin_1', options=dict(
    pos_x = f"{pos_x_cp1}um",
    pos_y = f"{pos_y_cp1}um",
    orientation = "270.0"
))

asym = 0
cpw1 = connect('cpw1', 'Qubit_1', 'connector_1', 'coupling_pin_1', 'open', '4037um', f'+{asym}um')

gui.rebuild()
gui.autoscale()



AttributeError: property 'format' of 'fast_csr_matrix' object has no setter